# SYNAPES Studio 2026 — Day 2 Multi-View 3D Worker (Kaggle 2×T4)

Notebook này dựng **Worker 3D GPU miễn phí (Kaggle T4 ×2)** chạy thật các mô hình **RMBG-2.0 → DINOv2 Consistency Gate → Hunyuan3D-2mv / TRELLIS.2 → UniRig Auto-Rig → `.GLB`** và cấp HTTPS Tunnel trực tiếp cho SYNAPES WebApp (BYOK).

- Yêu cầu Kaggle Secrets (Add-ons → Secrets): `HF_TOKEN` (bắt buộc nếu tải weight gated) và tùy chọn `R2_ENDPOINT_URL`, `R2_BUCKET`, `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`, `R2_PUBLIC_BASE_URL`.

In [ ]:
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}:', torch.cuda.get_device_name(i))

In [ ]:
!git clone --depth 1 https://github.com/nguyenhoangthienhvt15-bot/synapes-auto-tool.git /kaggle/working/synapes-auto-tool || true
!pip install -q fastapi uvicorn pydantic trimesh boto3 transformers huggingface_hub accelerate pillow numpy
%cd /kaggle/working/synapes-auto-tool/kaggle_workers

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    for key in ['HF_TOKEN', 'R2_ENDPOINT_URL', 'R2_BUCKET', 'R2_ACCESS_KEY_ID', 'R2_SECRET_ACCESS_KEY', 'R2_PUBLIC_BASE_URL']:
        try:
            val = _secrets.get_secret(key)
            if val:
                os.environ[key] = val
        except Exception:
            pass
except Exception:
    print('Kaggle Secrets not detected — reading from local environment.')
!python multiview_3d_worker.py --check

In [ ]:
# Chạy Worker ở cổng 8190 kèm Cloudflare Tunnel HTTPS (*.trycloudflare.com)
# Copy link https://....trycloudflare.com dán vào mục 'ComfyUI / Kaggle 2xT4 Worker' trên SYNAPES Studio
!python multiview_3d_worker.py --serve --port 8190 --tunnel